# Copositive QA-abstention — run on a real LLM (Qwen3-0.6B)

Certified (absolute-threshold) vs calibrated (softmax-confidence) abstention under distribution shift.
**GPU is preset. Runtime → Run all**, then paste the printed tables back. (Part 1 ≈10 min; Part 2/LoRA ≈20 min.)


In [ ]:
# 1. deps
!pip -q install -U transformers datasets

In [ ]:
# 2. fetch scripts (public repo, no auth)
!wget -q https://raw.githubusercontent.com/areidyOTH/copositive-colab/main/qa_abstention.py -O qa_abstention.py
print('got qa_abstention.py')

## Part 1 — frozen backbone (R14): train only a read head on the LLM's fixed embeddings


In [ ]:
# 3 seeds on Qwen3-0.6B (decoder -> last-token pooling). Fallback: Qwen/Qwen2.5-0.5B
MODEL = 'Qwen/Qwen3-0.6B'
for s in range(3):
    print(f'\n========== seed {s} ==========')
    !python qa_abstention.py --model {MODEL} --device cuda --pooling last --n_train 4000 --n_eval 2000 --steps 600 --seed {s}

## Part 2 — LoRA-adapt the backbone

Now let the **embeddings** adapt (small LoRA adapters trained inside the backbone + the head), not just
a head on frozen embeddings. Trains frozen AND LoRA, both reads, prints the comparison, and **saves all 4
models to `artifacts/` for you to download and compare locally.**


In [ ]:
# deps for LoRA
!pip -q install -U peft

In [ ]:
# fetch the LoRA script (and ensure the base script is present)
!wget -q https://raw.githubusercontent.com/areidyOTH/copositive-colab/main/qa_abstention.py -O qa_abstention.py
!wget -q https://raw.githubusercontent.com/areidyOTH/copositive-colab/main/qa_abstention_lora.py -O qa_abstention_lora.py
print('got qa_abstention_lora.py')

In [ ]:
# train frozen + LoRA (both reads), print comparison table, save artifacts/  (~15-20 min on T4)
!python qa_abstention_lora.py --model {MODEL} --device cuda --pooling last

In [ ]:
# zip the 4 trained models and download them to compare locally
!zip -qr artifacts.zip artifacts
from google.colab import files; files.download('artifacts.zip')
print('downloading artifacts.zip — frozen+LoRA heads + LoRA adapters for both reads')